# LDTF-BERT on AG News — Fine-tune vs Frozen Encoder

Notebook companion for the project in `src/`. Runs two experiments:

* **A**: Fine-tune the full BERT encoder + 3 custom routers (`label_queries`,
  `token_router`, `depth_router`) + `class_scorer`.
* **B**: Freeze the BERT encoder; train only the routers and scorer.

Cells are idempotent — re-running the notebook only rewrites the output
directory, never the source tree.


In [ ]:
"""Cell 1: install dependencies. Idempotent; safe to re-run."""
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                         "torch", "transformers", "scikit-learn",
                         "pandas", "numpy", "pyarrow", "matplotlib"])
print("[cell 1] dependencies ready")


## Cell 2 — Mount Drive and sync the project tree

Assumes the project lives at `MyDrive/HocSau_LDTF_BERT/` on Drive. The cell
creates a symlink `/content/HocSau_LDTF_BERT -> /content/drive/MyDrive/...`
so that all outputs land back on Drive automatically.


In [ ]:
"""Cell 2: mount Drive and symlink the project so checkpoints persist."""
import os, pathlib
from google.colab import drive

DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/HocSau_LDTF_BERT")
PROJECT_ROOT = pathlib.Path("/content/HocSau_LDTF_BERT")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        f"Project not found on Drive at {DRIVE_ROOT}. "
        "Copy the repo there first."
    )

drive.mount("/content/drive", force_remount=False)
if not PROJECT_ROOT.exists():
    os.symlink(DRIVE_ROOT, PROJECT_ROOT)
print(f"[cell 2] project root -> {PROJECT_ROOT}")
%cd {PROJECT_ROOT}
import sys; sys.path.insert(0, str(PROJECT_ROOT))


## Cell 3 — Sanity check the data

Load the parquet splits, print shapes and label distributions, and run a
tokenizer pass on a single batch.


In [ ]:
"""Cell 3: load parquet splits and verify tokenization works on one batch."""
import pandas as pd
from transformers import AutoTokenizer

from src import config
from src.dataset import load_split, AgNewsDataset

train_df = load_split(config.PROCESSED_TRAIN)
val_df   = load_split(config.PROCESSED_VAL)
test_df  = load_split(config.PROCESSED_TEST)
print(f"[cell 3] train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
print("[cell 3] label counts (train):", train_df['label'].value_counts().to_dict())

tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME, use_fast=True)
sample = AgNewsDataset(train_df.head(2), tokenizer=tokenizer)
print("[cell 3] sample[0] keys:", list(sample[0].__dict__.keys()))
print("[cell 3] input_ids shape:", sample[0].input_ids.shape)


## Cell 4 — Build the model and show parameter counts

Constructs `LdtfBert`, prints parameter counts per module, and runs one
forward pass on a dummy batch to confirm shapes are correct.


In [ ]:
"""Cell 4: build the model and confirm forward shapes."""
import torch
from src.models import LdtfBert

model = LdtfBert(model_name=config.MODEL_NAME, freeze_encoder=False)
counts = model.count_parameters()
for name, summary in counts.items():
    print(f"  {name:>14}: total={summary['total']:,} "
          f"trainable={summary['trainable']:,} frozen={summary['frozen']:,}")

dummy = {
    "input_ids":      torch.zeros(2, config.MAX_LENGTH, dtype=torch.long),
    "attention_mask": torch.ones (2, config.MAX_LENGTH, dtype=torch.long),
    "token_type_ids": torch.zeros(2, config.MAX_LENGTH, dtype=torch.long),
}
with torch.inference_mode():
    out = model(**dummy)
print("logits shape:", out["logits"].shape)
print("token_attention shape:", out["token_attention"].shape)
print("depth_attention shape:", out["depth_attention"].shape)


## Cell 5 — Experiment A: fine-tune (full backprop)

Trains the full model end-to-end on the train split, evaluates on val each
epoch, and writes `outputs/finetune/best_model.pt` + JSON metrics.


In [ ]:
"""Cell 5: fine-tune the full model (option A)."""
from src import config
from src.dataset import build_all_dataloaders
from src.evaluate import build_model_from_checkpoint, evaluate_checkpoint
from src.models import LdtfBert
from src.train import TrainConfig, train_model
from src.utils import set_seed

set_seed(config.SEED)
tokenizer = LdtfBert.build_tokenizer(config.MODEL_NAME)
loaders = build_all_dataloaders(tokenizer=tokenizer)

model = LdtfBert(model_name=config.MODEL_NAME, freeze_encoder=False)
cfg = TrainConfig(output_dir=config.FINETUNE_OUTPUT, freeze_encoder=False)
result = train_model(model=model, train_loader=loaders["train"],
                     val_loader=loaders["validation"], train_config=cfg)
print(f"[cell 5] best val acc={result.best_val_accuracy:.4f} @ epoch={result.best_epoch}")

eval_model = build_model_from_checkpoint(config.FINETUNE_OUTPUT / "best_model.pt")
test_metrics = evaluate_checkpoint(
    model=eval_model, dataloader=loaders["test"],
    checkpoint_path=config.FINETUNE_OUTPUT / "best_model.pt",
    output_path=config.FINETUNE_OUTPUT / "test_metrics.json",
)
print(f"[cell 5] test acc={test_metrics['accuracy']:.4f} "
      f"f1_macro={test_metrics['f1_macro']:.4f}")


## Cell 6 — Experiment B: frozen encoder

Same data, but only the routers and scorer receive gradient updates; BERT is
frozen.


In [ ]:
"""Cell 6: frozen-encoder experiment (option B)."""
from src import config
from src.dataset import build_all_dataloaders
from src.evaluate import build_model_from_checkpoint, evaluate_checkpoint
from src.models import LdtfBert
from src.train import TrainConfig, train_model
from src.utils import set_seed

set_seed(config.SEED)
tokenizer = LdtfBert.build_tokenizer(config.MODEL_NAME)
loaders = build_all_dataloaders(tokenizer=tokenizer)

model = LdtfBert(model_name=config.MODEL_NAME, freeze_encoder=True)
cfg = TrainConfig(output_dir=config.FROZEN_OUTPUT, freeze_encoder=True)
result = train_model(model=model, train_loader=loaders["train"],
                     val_loader=loaders["validation"], train_config=cfg)
print(f"[cell 6] best val acc={result.best_val_accuracy:.4f} @ epoch={result.best_epoch}")

eval_model = build_model_from_checkpoint(config.FROZEN_OUTPUT / "best_model.pt")
test_metrics = evaluate_checkpoint(
    model=eval_model, dataloader=loaders["test"],
    checkpoint_path=config.FROZEN_OUTPUT / "best_model.pt",
    output_path=config.FROZEN_OUTPUT / "test_metrics.json",
)
print(f"[cell 6] test acc={test_metrics['accuracy']:.4f} "
      f"f1_macro={test_metrics['f1_macro']:.4f}")


## Cell 7 — Build comparison table

Loads both `val_metrics.json` and `test_metrics.json` files and prints a
single comparison DataFrame.


In [ ]:
"""Cell 7: build the comparison DataFrame from JSON metrics."""
import pandas as pd
from src.utils import load_json

def _row(name, val_path, test_path):
    val = load_json(val_path)
    test = load_json(test_path)
    return {
        "experiment": name,
        "best_val_acc": val["best_val_accuracy"],
        "best_epoch": val["best_epoch"],
        "test_acc": test["accuracy"],
        "test_f1_macro": test["f1_macro"],
        "test_f1_weighted": test["f1_weighted"],
    }

rows = [
    _row("finetune",
         config.FINETUNE_OUTPUT / "val_metrics.json",
         config.FINETUNE_OUTPUT / "test_metrics.json"),
    _row("frozen",
         config.FROZEN_OUTPUT  / "val_metrics.json",
         config.FROZEN_OUTPUT  / "test_metrics.json"),
]
df = pd.DataFrame(rows)
print(df.to_string(index=False))


## Cell 8 — Plot training curves

Two side-by-side panels: training loss and validation accuracy across epochs
for both experiments.


In [ ]:
"""Cell 8: plot training/validation curves for both experiments."""
import matplotlib.pyplot as plt
from src.utils import load_json

history = {
    "finetune": load_json(config.FINETUNE_OUTPUT / "val_metrics.json")["history"],
    "frozen":   load_json(Config.FROZEN_OUTPUT  / "val_metrics.json")["history"],
}
if "Config" not in dir():
    from src import config as Config
history = {
    "finetune": load_json(Config.FINETUNE_OUTPUT / "val_metrics.json")["history"],
    "frozen":   load_json(Config.FROZEN_OUTPUT  / "val_metrics.json")["history"],
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in history.items():
    epochs = [r["epoch"] for r in hist]
    axes[0].plot(epochs, [r["train_loss"] for r in hist], marker="o", label=name)
    axes[1].plot(epochs, [r["val_accuracy"] for r in hist], marker="o", label=name)
axes[0].set(title="Training Loss", xlabel="Epoch", ylabel="Loss"); axes[0].legend(); axes[0].grid(True, ls="--", alpha=.5)
axes[1].set(title="Validation Accuracy", xlabel="Epoch", ylabel="Accuracy"); axes[1].legend(); axes[1].grid(True, ls="--", alpha=.5)
fig.tight_layout()
fig.savefig(Config.FIGURES_DIR / "loss_curve.png", dpi=150)
plt.show()


## Cell 9 — Confusion matrices

One row × two columns: confusion matrices for fine-tune vs frozen on the
test split.


In [ ]:
"""Cell 9: render confusion matrices for both experiments."""
import numpy as np
import matplotlib.pyplot as plt
from src.utils import load_json

tests = {
    "finetune": load_json(config.FINETUNE_OUTPUT / "test_metrics.json"),
    "frozen":   load_json(config.FROZEN_OUTPUT  / "test_metrics.json"),
}
fig, axes = plt.subplots(1, len(tests), figsize=(5 * len(tests), 4))
if len(tests) == 1: axes = [axes]
for ax, (name, metrics) in zip(axes, tests.items()):
    matrix = np.asarray(metrics["confusion_matrix"])
    ax.imshow(matrix, cmap="Blues")
    ax.set(title=f"Confusion: {name}",
           xticks=range(len(config.LABEL_NAMES)), xticklabels=config.LABEL_NAMES,
           yticks=range(len(config.LABEL_NAMES)), yticklabels=config.LABEL_NAMES,
           xlabel="Predicted", ylabel="True")
    ax.set_xticklabels(config.LABEL_NAMES, rotation=30, ha="right")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, int(matrix[i, j]), ha="center", va="center", color="black")
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "confusion.png", dpi=150)
plt.show()


## Cell 10 — Persist reports to Drive

Writes `comparison.{csv,md,json}` and the PNGs to `reports/` (which already
lives on Drive via the symlink from cell 2).


In [ ]:
"""Cell 10: persist CSV/Markdown comparison + figures to reports/."""
import pandas as pd
from src.utils import load_json, save_json

rows = [
    {
        "experiment": "finetune",
        "best_val_accuracy": load_json(config.FINETUNE_OUTPUT / "val_metrics.json")["best_val_accuracy"],
        "best_epoch": load_json(config.FINETUNE_OUTPUT / "val_metrics.json")["best_epoch"],
        "test_accuracy": load_json(config.FINETUNE_OUTPUT / "test_metrics.json")["accuracy"],
        "test_f1_macro": load_json(config.FINETUNE_OUTPUT / "test_metrics.json")["f1_macro"],
        "test_f1_weighted": load_json(config.FINETUNE_OUTPUT / "test_metrics.json")["f1_weighted"],
    },
    {
        "experiment": "frozen",
        "best_val_accuracy": load_json(config.FROZEN_OUTPUT / "val_metrics.json")["best_val_accuracy"],
        "best_epoch": load_json(config.FROZEN_OUTPUT / "val_metrics.json")["best_epoch"],
        "test_accuracy": load_json(config.FROZEN_OUTPUT / "test_metrics.json")["accuracy"],
        "test_f1_macro": load_json(config.FROZEN_OUTPUT / "test_metrics.json")["f1_macro"],
        "test_f1_weighted": load_json(config.FROZEN_OUTPUT / "test_metrics.json")["f1_weighted"],
    },
]
df = pd.DataFrame(rows)
df.to_csv(config.REPORTS_DIR / "comparison.csv", index=False)
(config.REPORTS_DIR / "comparison.md").write_text(
    df.to_markdown(index=False, floatfmt=".4f"), encoding="utf-8")
save_json({"summary": df.to_dict(orient="records")}, config.REPORTS_DIR / "comparison.json")
print("[cell 10] wrote", config.REPORTS_DIR / "comparison.csv")
print("[cell 10] figures:", list(config.FIGURES_DIR.iterdir()))
